## Preparing data and notebook

In [1]:
import os
import numpy as np
import pandas as pd
import mne
from scipy import signal
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.constraints import max_norm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from tensorflow.keras.constraints import max_norm
import plotly.graph_objects as go
import plotly.figure_factory as ff
from plotly.subplots import make_subplots

In [2]:
class Config:
    BASE_PATH = 'C:/ESC5Work/ProjectM5/files'
    SAMPLING_RATE = 160 # Hz
    # RESAMPLE = 256
    N_CHANNELS = 64 # Number of EEG channels
    N_CLASSES = 2 # Left vs Right hand imagery

    ALL_SUBJECTS = list(range(1, 110))
    TRAIN_SUBJECTS = list(range(1, 78)) # Subjects 1-77 for training (approx. 70%)
    VALID_SUBJECTS = list(range(78, 94)) # Subjects 78-93 for validation (approx. 15%)
    TEST_SUBJECTS = list(range(94, 110)) # Subjects 94-109 for final testing (approx. 15%)

    # Signal Processing Parameters
    LOWCUT = 8.0 # Lower cutoff frequency (Hz) for Mu 8-12 rhythm
    HIGHCUT = 30.0 # Upper cutoff frequency (Hz) for Beta 13-30 rhythm
    WINDOW_SIZE = 4.0 # Time window in seconds (full imagery period)
    T_MIN = 0.0 # Start time of the epoch relative to the event
    T_MAX = 4.0 # End time of the epoch

    # EEGNet Architecture Parameters
    F1 = 8 #8 # Number of temporal filters
    D = 2 #2 # Number of spatial filters
    F2 = 16 #16 # Number of pointwise filters
    KERNEL_LENGTH = 64 # Length of the temporal convolution kernel
    DROPOUT_RATE = 0.5 # Dropout rate for regularization

    # Training Parameters
    BATCH_SIZE = 16 # Smaller batch size for better generalization on EEG data
    EPOCHS = 200 # Max epochs, with early stopping
    LEARNING_RATE = 0.0005 #0.001
    EARLY_STOPPING_PATIENCE = 30 # Increased patience for EEGNet

    # Visualization with Plotly
    PLOTLY_TEMPLATE = 'plotly_dark'
    
    SELECTED_CHANNELS = ['Fc5.', 'Fc6.', 'Af3.', 'Af4.', 'F7..', 'F3..', 'F4..', 'F8..', 'T7..', 'T8..', 'P7..', 'P8..', 'O1..', 'O2..']

config = Config()

# Preprocessing

In [3]:
import warnings
warnings.filterwarnings(
    "ignore", 
    message=".*All epochs were dropped.*", 
    category=RuntimeWarning
)

In [4]:
def load_and_preprocess_subject_data(subject_id, base_path):
    mne.set_log_level("ERROR")
    mi_runs = [4, 8, 12]

    raw_files = []
    for run_id in mi_runs:
        subject_str = f"S{subject_id:03d}"
        run_str = f"R{run_id:02d}"
        file_path = f"{base_path}/{subject_str}/{subject_str}{run_str}.edf"

        if os.path.exists(file_path):
            try:
                raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)
                raw_files.append(raw)
            except Exception as e:
                print(f"-- error to load {file_path}: {e}")
        else:
            pass

    if not raw_files:
        return None, None

    raw_concat = mne.concatenate_raws(raw_files, verbose=False)

    raw_concat.filter(config.LOWCUT, config.HIGHCUT, fir_design='firwin', skip_by_annotation='edge', verbose=False)
    raw_concat.notch_filter(freqs=50, verbose=False)
    # raw_concat.resample(config.RESAMPLE, npad="auto")
    raw_concat.set_eeg_reference('average', projection=False)
    raw_concat.apply_proj(verbose=False)

    events, event_id = mne.events_from_annotations(raw_concat, verbose=False)
    # print(events)

    # picks = mne.pick_types(raw_concat.info, meg=False, eeg=True, stim=False, eog=False, exclude='bads')
    picks = mne.pick_channels(raw_concat.info["ch_names"], include=config.SELECTED_CHANNELS)
    
    # Epoching with rejection
    reject_criteria = dict(eeg=200e-6)  
    flat_criteria = dict(eeg=1e-6)
    
    try:
        epochs = mne.Epochs(raw_concat, events, dict(left_hand=2, right_hand=3),
                            tmin=config.T_MIN, tmax=config.T_MAX, proj=True,
                            picks=picks, baseline=None, preload=True, verbose=False, reject=reject_criteria, flat=flat_criteria,
                            on_missing='warn')
    except ValueError:
        return None, None
    
    # Normalize data
    X = epochs.get_data()
    X = (X - X.mean(axis=-1, keepdims=True)) / X.std(axis=-1, keepdims=True)

    labels = epochs.events[:, -1] - 2  # Map labels to 0 (left) and 1 (right)

    return X, labels

In [5]:
def load_dataset_for_subjects(subject_list, base_path):
    all_epochs_list = []
    all_labels_list = []
    expected_n_samples = None

    for subject_id in subject_list:
        print(f"Processing {subject_id:03d}...", end=" -> ")
        epochs, labels = load_and_preprocess_subject_data(subject_id, config.BASE_PATH)

        if epochs is not None and len(epochs) > 0:
            if expected_n_samples is None:
                expected_n_samples = epochs.shape[2]
            if epochs.shape[2] != expected_n_samples:
                print(f"-- Wrong length ({epochs.shape[2]}). Skipping subject.")
                continue

            all_epochs_list.append(epochs)
            all_labels_list.append(labels)
            print(f"Finish {len(epochs)} trials.")
        else:
            print("-- No valid trials found. (dropped all)")

    if not all_epochs_list:
        return np.array([]), np.array([])

    return np.concatenate(all_epochs_list, axis=0), np.concatenate(all_labels_list, axis=0)

In [13]:
print("\n--- Loading Training Data ---")
X_train, y_train = load_dataset_for_subjects(config.TRAIN_SUBJECTS, config.BASE_PATH)

print("\n--- Loading Validation Data ---")
X_val, y_val = load_dataset_for_subjects(config.VALID_SUBJECTS, config.BASE_PATH)

if X_train.size > 0 and X_val.size > 0:
    print("\n Data Summary >> ")
    print(f" Training data shape: {X_train.shape}")
    print(f" Training labels shape: {y_train.shape} (Left: {np.sum(y_train == 0)}, Right: {np.sum(y_train == 1)})\")")
    print(f" Validation data shape: {X_val.shape}")
    print(f" Validation labels shape: {y_val.shape} (Left: {np.sum(y_val == 0)}, Right: {np.sum(y_val == 1)})\")")
else:
    print("\n-- Error")


--- Loading Training Data ---
Processing 001... -> Finish 41 trials.
Processing 002... -> Finish 45 trials.
Processing 003... -> Finish 41 trials.
Processing 004... -> Finish 45 trials.
Processing 005... -> Finish 45 trials.
Processing 006... -> Finish 41 trials.
Processing 007... -> Finish 39 trials.
Processing 008... -> Finish 45 trials.
Processing 009... -> -- No valid trials found. (dropped all)
Processing 010... -> Finish 42 trials.
Processing 011... -> Finish 44 trials.
Processing 012... -> Finish 45 trials.
Processing 013... -> -- No valid trials found. (dropped all)
Processing 014... -> Finish 45 trials.
Processing 015... -> Finish 28 trials.
Processing 016... -> Finish 45 trials.
Processing 017... -> Finish 8 trials.
Processing 018... -> Finish 45 trials.
Processing 019... -> Finish 16 trials.
Processing 020... -> Finish 45 trials.
Processing 021... -> Finish 45 trials.
Processing 022... -> Finish 35 trials.
Processing 023... -> Finish 16 trials.
Processing 024... -> -- No va

# EEGNET (CNN)

## Prepare DATA

In [7]:
def prep_eegnetDATA(X_train, y_train, X_val, y_val):
    X_train_flat = X_train.reshape(X_train.shape[0], -1)
    X_val_flat = X_val.reshape(X_val.shape[0], -1)
    
    scaler = StandardScaler()
    X_train_FitFlat = scaler.fit_transform(X_train_flat)
    X_val_FitFlat = scaler.fit_transform(X_val_flat)
    
    X_train_fit = X_train_FitFlat.reshape(X_train.shape)
    X_val_fit = X_val_FitFlat.reshape(X_val.shape)
    
    X_train_fin = X_train_fit[:, :, :, np.newaxis]
    X_val_fin = X_val_fit[:, :, :, np.newaxis]
    y_train_fin = to_categorical(y_train, config.N_CLASSES)
    y_val_fin = to_categorical(y_val, config.N_CLASSES)
    
    return X_train_fin, y_train_fin, X_val_fin, y_val_fin, scaler

In [14]:
X_train_fin, y_train_fin, X_val_fin, y_val_fin, scaler = prep_eegnetDATA(X_train, y_train, X_val, y_val)

print(f'Train shape (X): {X_train_fin.shape}')
print(f'Validation shape (X): {X_val_fin.shape}')
# trial, channel, sample

Train shape (X): (2757, 14, 641, 1)
Validation shape (X): (530, 14, 641, 1)


## Prepare Model

In [17]:
def EEGNet_model(nb_classes, Chans=config.N_CHANNELS, Samples=X_train_fin.shape[2], dropoutRate=0.5,
           kernLength=64, F1=config.F1, D=config.D, F2=config.F2, norm_rate=0.25):
    input_main = layers.Input(shape=(Chans, Samples, 1), name='input_layer')

    # --- Block 1 ---
    # Temporal Convolution
    block1 = layers.Conv2D(F1, (1, kernLength), padding='same', use_bias=False)(input_main)
    block1 = layers.BatchNormalization()(block1)

    # Depthwise Spatial Convolution
    block1 = layers.DepthwiseConv2D((Chans, 1), use_bias=False, depth_multiplier=D,
                                     depthwise_constraint=max_norm(1.))(block1)
    block1 = layers.BatchNormalization()(block1)
    block1 = layers.Activation('elu')(block1)
    block1 = layers.AveragePooling2D((1, D))(block1)
    block1 = layers.Dropout(dropoutRate)(block1)

    # --- Block 2 ---
    # Separable Convolution
    block2 = layers.SeparableConv2D(F2, (1, F2), use_bias=False, padding='same')(block1)
    block2 = layers.BatchNormalization()(block2)
    block2 = layers.Activation('elu')(block2)
    block2 = layers.AveragePooling2D((1, F1))(block2)
    block2 = layers.Dropout(dropoutRate)(block2)

    # --- Classification Block ---
    flatten = layers.Flatten(name='flatten')(block2)
    dense = layers.Dense(nb_classes, name='dense', kernel_constraint=max_norm(norm_rate))(flatten)
    softmax = layers.Activation('softmax', name='softmax')(dense)

    return models.Model(inputs=input_main, outputs=softmax, name="EEGNet")

# --- Instantiate and Compile the Model ---
input_shape = X_train_fin.shape[1:]

model = EEGNet_model(
    nb_classes=config.N_CLASSES,
    Chans=input_shape[0],
    Samples=input_shape[1],
    dropoutRate=config.DROPOUT_RATE,
    kernLength=config.KERNEL_LENGTH,
    F1=config.F1, D=config.D, F2=config.F2
)

# Compile the model with Adam optimizer
optimizer = optimizers.Adam(learning_rate=config.LEARNING_RATE)
model.compile(loss='categorical_crossentropy', optimizer=optimizer, metrics=['accuracy'])

# Print the model summary
print("EEGNet Model Summary:")
model.summary()

EEGNet Model Summary:


Model: "EEGNet"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 14, 641, 1)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 14, 641, 8)     │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 14, 641, 8)     │            32 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv2d_2              │ (None, 1, 641, 16)     │           224 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 1, 641, 16)     │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_4 (Activation)       │ (None, 1, 641, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_4             │ (None, 1, 320, 16)     │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 1, 320, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ separable_conv2d_2              │ (None, 1, 320, 16)     │           512 │
│ (SeparableConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 1, 320, 16)     │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_5 (Activation)       │ (None, 1, 320, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_5             │ (None, 1, 40, 16)      │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 1, 40, 16)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 640)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 2)              │         1,282 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ softmax (Activation)            │ (None, 2)              │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,690 (10.51 KB)

 Trainable params: 2,610 (10.20 KB)

 Non-trainable params: 80 (320.00 B)

## Train Model

In [18]:
training_callbacks = [
    # callbacks.EarlyStopping(
    #     monitor='val_accuracy',
    #     patience=config.EARLY_STOPPING_PATIENCE,
    #     restore_best_weights=True,
    #     verbose=1,
    #     mode='max'
    # ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=15, # Increased patience for learning rate reduction
        min_lr=1e-7,
        verbose=1
    ),
    callbacks.ModelCheckpoint(
        'best_eegnet_model.keras',
        monitor='val_accuracy', 
        save_best_only=True,
        verbose=1,
        mode='max'
    )
]

history = model.fit( # fit the EEGNet model
    X_train_fin,
    y_train_fin,
    batch_size=config.BATCH_SIZE,
    epochs=109,
    validation_data=(X_val_fin, y_val_fin),
    callbacks=training_callbacks,
    verbose=1,
    shuffle=True
)

Epoch 1/109
173/173 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.4855 - loss: 0.6969
Epoch 1: val_accuracy improved from -inf to 0.45849, saving model to best_eegnet_model.keras
173/173 ━━━━━━━━━━━━━━━━━━━━ 12s 47ms/step - accuracy: 0.4855 - loss: 0.6969 - val_accuracy: 0.4585 - val_loss: 0.6935 - learning_rate: 5.0000e-04
Epoch 2/109
172/173 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.5255 - loss: 0.6910
Epoch 2: val_accuracy improved from 0.45849 to 0.52642, saving model to best_eegnet_model.keras
173/173 ━━━━━━━━━━━━━━━━━━━━ 9s 54ms/step - accuracy: 0.5254 - loss: 0.6911 - val_accuracy: 0.5264 - val_loss: 0.6936 - learning_rate: 5.0000e-04
Epoch 3/109
173/173 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.5164 - loss: 0.6942
Epoch 3: val_accuracy did not improve from 0.52642
173/173 ━━━━━━━━━━━━━━━━━━━━ 7s 41ms/step - accuracy: 0.5165 - loss: 0.6942 - val_accuracy: 0.5000 - val_loss: 0.6962 - learning_rate: 5.0000e-04
Epoch 4/109
173/173 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - a

# Evaluation

In [19]:
# --- Plot Training History ---
def plot_training_history(history):
    fig = make_subplots(rows=1, cols=2, subplot_titles=("Model Accuracy", "Model Loss"))

    # Accuracy Plot
    fig.add_trace(go.Scatter(y=history.history['accuracy'], name='Train Accuracy', mode='lines+markers'), row=1, col=1)
    fig.add_trace(go.Scatter(y=history.history['val_accuracy'], name='Validation Accuracy', mode='lines+markers'), row=1, col=1)

    # Loss Plot
    fig.add_trace(go.Scatter(y=history.history['loss'], name='Train Loss', mode='lines+markers'), row=1, col=2)
    fig.add_trace(go.Scatter(y=history.history['val_loss'], name='Validation Loss', mode='lines+markers'), row=1, col=2)

    fig.update_layout(title_text="Model Training History", template=config.PLOTLY_TEMPLATE, height=400)
    fig.update_xaxes(title_text="Epoch")
    fig.update_yaxes(title_text="Metric Value")
    fig.show()

print("\n--- Training History ---")
plot_training_history(history)

# --- Validation Set Evaluation ---
print("\n--- Evaluating Performance on Val Set ---")
best_model = models.load_model('best_eegnet_model.keras')
y_pred_val = best_model.predict(X_val_fin, batch_size=config.BATCH_SIZE)
y_pred_classes = np.argmax(y_pred_val, axis=1)
y_true_classes = np.argmax(y_val_fin, axis=1)

# Classification Report
print("\nClassification Report (Val Set):")
class_report = classification_report(y_true_classes, y_pred_classes, target_names=['Left Hand', 'Right Hand'])
print(class_report)

# Confusion Matrix
def plot_confusion_matrix(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    labels = ['Left Hand', 'Right Hand']
    # Add annotations as text
    annotations = []
    for i, row in enumerate(cm):
        for j, value in enumerate(row):
            annotations.append(
                dict(x=labels[j], y=labels[i], text=str(value), showarrow=False)
            )

    fig = go.Figure(data=go.Heatmap(
                    z=cm,
                    x=labels,
                    y=labels,
                    colorscale='Blues'
                ))

    fig.update_layout(title_text="Confusion Matrix (Val Set)",
                      xaxis_title="Predicted Label", yaxis_title="True Label",
                      template=config.PLOTLY_TEMPLATE, annotations=annotations)
    fig.show()

print("\n--- Confusion Matrix ---")
plot_confusion_matrix(y_true_classes, y_pred_classes)


--- Training History ---



--- Evaluating Performance on Val Set ---
34/34 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step

Classification Report (Val Set):
              precision    recall  f1-score   support

   Left Hand       0.70      0.76      0.73       266
  Right Hand       0.74      0.67      0.70       264

    accuracy                           0.72       530
   macro avg       0.72      0.72      0.72       530
weighted avg       0.72      0.72      0.72       530


--- Confusion Matrix ---


## Test

In [20]:
print("\nLoading test data -> ")
X_test, y_test = load_dataset_for_subjects(config.TEST_SUBJECTS, config.BASE_PATH)

print(f"Success. Found {X_test.shape[0]} trials.")
print(f"\nTest data shape: {X_test.shape}")
print(f", Test labels shape: {y_test.shape} (Left: {np.sum(y_test == 0)}, Right: {np.sum(y_test == 1)})")

print("\nPreparing test data ->")
X_test_flat = X_test.reshape(X_test.shape[0], -1)
X_test_scaled_flat = scaler.transform(X_test_flat)  # IMPORTANT: Only transform, do not fit again
X_test_scaled = X_test_scaled_flat.reshape(X_test.shape)
X_test_final = X_test_scaled[:, :, :, np.newaxis]
y_test_final = to_categorical(y_test, config.N_CLASSES)
print(f" Final Test Shape: {X_test_final.shape}")

print("\nEvaluating model on the test set :")
test_loss, test_accuracy = best_model.evaluate(X_test_final, y_test_final, verbose=0, batch_size=config.BATCH_SIZE)
print(f"\nFinal Test Accuracy: {test_accuracy*100:.2f}%")
print(f"   Final Test Loss: {test_loss:.4f}")

y_pred_test = best_model.predict(X_test_final, batch_size=config.BATCH_SIZE)
y_pred_test_classes = np.argmax(y_pred_test, axis=1)
y_true_test_classes = y_test

print("\nClassification Report (Test Set):")
print(classification_report(y_true_test_classes, y_pred_test_classes, target_names=['Left Hand', 'Right Hand']))

def plot_test_confusion_matrix(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    labels = ['Left Hand', 'Right Hand']
    fig = ff.create_annotated_heatmap(z=cm, x=labels, y=labels, colorscale='Greens')
    fig.update_layout(title_text="Confusion Matrix (Final Test Set)",
                    xaxis_title="Predicted Label", yaxis_title="True Label",
                    template=config.PLOTLY_TEMPLATE)
    fig.show()

plot_test_confusion_matrix(y_true_test_classes, y_pred_test_classes)


Loading test data -> 
Processing 094... -> Finish 26 trials.
Processing 095... -> Finish 7 trials.
Processing 096... -> Finish 18 trials.
Processing 097... -> Finish 45 trials.
Processing 098... -> Finish 45 trials.
Processing 099... -> Finish 3 trials.
Processing 100... -> 

C:\Users\scpwk\AppData\Local\Temp\ipykernel_11876\3364080230.py:13: RuntimeWarning:

Limited 1 annotation(s) that were expanding outside the data range.

C:\Users\scpwk\AppData\Local\Temp\ipykernel_11876\3364080230.py:13: RuntimeWarning:

Limited 1 annotation(s) that were expanding outside the data range.

C:\Users\scpwk\AppData\Local\Temp\ipykernel_11876\3364080230.py:13: RuntimeWarning:

Limited 1 annotation(s) that were expanding outside the data range.



-- Wrong length (513). Skipping subject.
Processing 101... -> Finish 45 trials.
Processing 102... -> Finish 42 trials.
Processing 103... -> Finish 14 trials.
Processing 104... -> Finish 42 trials.
Processing 105... -> Finish 45 trials.
Processing 106... -> Finish 37 trials.
Processing 107... -> Finish 37 trials.
Processing 108... -> Finish 7 trials.
Processing 109... -> Finish 44 trials.
Success. Found 457 trials.

Test data shape: (457, 14, 641)
, Test labels shape: (457,) (Left: 229, Right: 228)

Preparing test data ->
 Final Test Shape: (457, 14, 641, 1)

Evaluating model on the test set :

Final Test Accuracy: 66.52%
   Final Test Loss: 0.5846
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step

Classification Report (Test Set):
              precision    recall  f1-score   support

   Left Hand       0.66      0.68      0.67       229
  Right Hand       0.67      0.65      0.66       228

    accuracy                           0.67       457
   macro avg       0.67      0.67      0.67       45